# Responses API streaming in Jupyter

This notebook shows the compact default transcript, category filtering, silent consumption, and opt-in protocol details. The examples force Web Search and Code Interpreter so their progress events and streamed Python are easy to see.

Install `yhelpers`, then set lowercase `folder_id` and `api_key` environment variables.

In [1]:
import sys
sys.path.append("../../src")
import os

from openai import OpenAI
from yhelpers.responses.streaming import jstream

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)
model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8"

## Web Search with the compact default

With no flags, `jstream` shows useful activity—such as “Web search”, the query, citations, and the answer—but hides response lifecycle bookkeeping, token usage, and diagnostic JSON.

In [2]:
web_stream = client.responses.create(
    model=model,
    instructions=(
        "You must use web search. Cite the source URL and clearly separate "
        "the verified fact from your explanation."
    ),
    input=(
        "Find the title currently shown on the Python 3 documentation home "
        "page. Then explain in one sentence what that page is for."
    ),
    tools=[{"type": "web_search", "search_context_size": "low"}],
    stream=True,
)
web_response = jstream(web_stream)
print("Response:", web_response.id, web_response.status)

<style>.yhelpers-stream-123086bffd0-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-123086bffd0-text' style='display:none'></div>

The title currently shown on the Python 3 documentation home page is "3.14.7 Documentation" [1].

This page serves as the official reference and guide for Python 3.14.7, providing comprehensive documentation including tutorials, library references, and language specifications.

[1] https://docs.python.org/3/index.html

Response: 87ce1537-cc6b-409d-befe-8ab92279ac3a completed


## Code Interpreter with selected categories

This call shows only answer text, Python execution, and errors. Generated Python remains complete and is finalized as a syntax-highlighted `python` fence.

In [3]:
code_stream = client.responses.create(
    model=model,
    instructions=(
        "You must use Code Interpreter. Show the exact Python code, report "
        "the numerical result, and explain the formula in one sentence."
    ),
    input="Calculate the sum of the squares from 1 through 100 with Python.",
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto"},
        }
    ],
    stream=True,
)
code_response = jstream(
    code_stream,
    events={"text", "code", "errors"},
)
print("Response:", code_response.id, code_response.status)

<style>.yhelpers-stream-12307673a50-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-12307673a50-text' style='display:none'></div>

I'll calculate the sum of squares from 1 through 100 using Python.

The formula for this calculation is: Σ(i²) from i=1 to 100

Let me execute this calculation:


<style>.yhelpers-stream-12307673a50-code-interpreter ~ *{color:#7c3aed;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-12307673a50-code-interpreter' style='display:none'></div>

**Code Interpreter**

```python
try:
    # Calculate the sum of squares from 1 to 100
    sum_of_squares = sum(i**2 for i in range(1, 101))
    
    # Print the result
    print(f"Sum of squares from 1 to 100: {sum_of_squares}")
    
    # The formula being used is the sum of i^2 for i from 1 to 100
    print("This calculates Σ(i²) where i ranges from 1 to 100")
    
except Exception as e:
    print(f"Error: {str(e)}")
```

<style>.yhelpers-stream-12307673a50-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-12307673a50-text' style='display:none'></div>

Sum of squares from 1 to 100: 338350

The calculation computes Σ(i²) for i ranging from 1 to 100, which equals 338350.

Response: aa4d2bee-6f6f-4427-9cc6-2680cea687ff completed


## Silent consumption

An empty event collection still consumes the complete stream and returns the final `Response`, but displays nothing while it runs.

In [4]:
silent_stream = client.responses.create(
    model=model,
    input="Reply with exactly: stream consumed",
    stream=True,
)
silent_response = jstream(silent_stream, events=[])
print(silent_response.output_text)

stream consumed


## Full protocol diagnostics (opt in)

`events="all"` includes lifecycle and usage events. `show_details=True` adds event names and bounded JSON payloads. Use this combination when diagnosing SDK or provider behavior; it is intentionally more verbose than the default.

In [5]:
debug_stream = client.responses.create(
    model=model,
    instructions="Use web search once, then answer with one sourced sentence.",
    input="What is the official Python documentation URL?",
    tools=[{"type": "web_search", "search_context_size": "low"}],
    stream=True,
)
debug_response = jstream(
    debug_stream,
    events="all",
    show_details=True,
    max_chars=1000,
)
print("Debug response:", debug_response.id)

<style>.yhelpers-stream-1230a30b3d0-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-1230a30b3d0-text' style='display:none'></div>

Официальная документация Python доступна по адресу https://docs.python.org/.

Debug response: c04859ab-0203-46ee-997d-5072766eca97


In [ ]:
client.close()